# Interactive plots: Acceleration RMS vs TIMESTAMP

This notebook scans `data/*.csv` and plots only `Acceleration RMS` against `TIMESTAMP` for each CSV.

Zoom and pan are enabled via `%matplotlib notebook`.

In [ ]:
%matplotlib notebook

from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd

print('Jupyter cwd:', os.getcwd(), flush=True)

DATA_DIR = Path('data')
print('DATA_DIR exists:', DATA_DIR.exists(), flush=True)

csv_files = sorted(DATA_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV file(s) in {DATA_DIR.resolve()}', flush=True)

for f in csv_files:
    print('-', f.name, flush=True)


In [ ]:
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR.resolve()}')

required_cols = {'TIMESTAMP', 'Acceleration RMS'}

for csv_path in csv_files:
    print(f'Plotting: {csv_path.name}', flush=True)
    df = pd.read_csv(csv_path)

    missing = required_cols - set(df.columns)
    if missing:
        print(f'  Skipping {csv_path.name}: missing columns {sorted(missing)}', flush=True)
        continue

    ts = pd.to_datetime(df['TIMESTAMP'], errors='coerce', dayfirst=True)
    y = pd.to_numeric(df['Acceleration RMS'], errors='coerce')
    valid = ts.notna() & y.notna()

    if valid.sum() == 0:
        print(f'  Skipping {csv_path.name}: no valid TIMESTAMP/Acceleration RMS rows', flush=True)
        continue

    ts = ts[valid]
    y = y[valid]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(ts, y, linewidth=1.0)
    ax.set_title(f'{csv_path.name} - Acceleration RMS vs TIMESTAMP')
    ax.set_xlabel('TIMESTAMP')
    ax.set_ylabel('Acceleration RMS')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


In [2]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from pathlib import Path

# Open interactive plots in browser
pio.renderers.default = "browser"

# ============================================
# Folder containing CSV files
# ============================================
folder_path = r"data/"

# Get all CSV files
csv_files = list(Path(folder_path).glob("*.csv"))

# ============================================
# Plot each CSV separately
# ============================================
for file in csv_files:

    print(f"Plotting: {file.name}")

    # Read CSV
    df = pd.read_csv(file)

    # Convert timestamp column
    df["TIMESTAMP"] = pd.to_datetime(
        df["TIMESTAMP"],
        dayfirst=True,
        format="mixed"
    )

    # Create figure
    fig = px.line(
        df,
        x="TIMESTAMP",
        y="Acceleration RMS",
        title=file.stem
    )

    # Layout
    fig.update_layout(
        xaxis_title="Timestamp",
        yaxis_title="Acceleration RMS",
        hovermode="x unified",
        template="plotly_white"
    )

    # Show separate interactive plot
    fig.show()

Plotting: AHU_2_9_Blower_DE_A.csv
Plotting: AHU_2_9_Blower_DE_V.csv
Plotting: AHU_2_9_Blower_DE_Vibration_X.csv
Plotting: AHU_2_9_Blower_NDE_A.csv
Plotting: AHU_2_9_Blower_NDE_H.csv
Plotting: AHU_2_9_Blower_NDE_V.csv
Plotting: AHU_2_9_motor_DE_H.csv
Plotting: AHU_2_9_motor_NDE_H.csv


In [1]:
# Smoothed replots (rolling mean) — same CSVs as above, less high-frequency noise
import pandas as pd
import plotly.express as px
import plotly.io as pio
from pathlib import Path

pio.renderers.default = "browser"

folder_path = r"data/"
csv_files = sorted(Path(folder_path).glob("*.csv"))

OUT_DIR = Path(folder_path).resolve() / "smoothed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Larger window = smoother curve (try 24–200 depending on sample density).
ROLL_WINDOW = 200

for file in csv_files:
    print(f"Smoothed plot: {file.name}")

    df = pd.read_csv(file)
    df["TIMESTAMP"] = pd.to_datetime(
        df["TIMESTAMP"],
        dayfirst=True,
        format="mixed",
    )
    df = df.sort_values("TIMESTAMP")

    y = pd.to_numeric(df["Acceleration RMS"], errors="coerce")
    y_smooth = y.rolling(window=ROLL_WINDOW, center=True, min_periods=1).mean()

    plot_df = df.assign(**{"Acceleration RMS (smoothed)": y_smooth})

    fig = px.line(
        plot_df,
        x="TIMESTAMP",
        y="Acceleration RMS (smoothed)",
        title=f"{file.stem} — smoothed (rolling mean, window={ROLL_WINDOW})",
    )
    fig.update_layout(
        xaxis_title="Timestamp",
        yaxis_title="Acceleration RMS (smoothed)",
        hovermode="x unified",
        template="plotly_white",
    )
    fig.show()

    out_path = OUT_DIR / f"{file.stem}_smoothed_roll{ROLL_WINDOW}.csv"
    plot_df.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}")

Smoothed plot: AHU_2_9_Blower_DE_A.csv
  Saved: C:\Users\NGYX\Desktop\Murata_NGYX\data\smoothed\AHU_2_9_Blower_DE_A_smoothed_roll200.csv
Smoothed plot: AHU_2_9_Blower_DE_V.csv
  Saved: C:\Users\NGYX\Desktop\Murata_NGYX\data\smoothed\AHU_2_9_Blower_DE_V_smoothed_roll200.csv
Smoothed plot: AHU_2_9_Blower_DE_Vibration_X.csv
  Saved: C:\Users\NGYX\Desktop\Murata_NGYX\data\smoothed\AHU_2_9_Blower_DE_Vibration_X_smoothed_roll200.csv
Smoothed plot: AHU_2_9_Blower_NDE_A.csv
  Saved: C:\Users\NGYX\Desktop\Murata_NGYX\data\smoothed\AHU_2_9_Blower_NDE_A_smoothed_roll200.csv
Smoothed plot: AHU_2_9_Blower_NDE_H.csv
  Saved: C:\Users\NGYX\Desktop\Murata_NGYX\data\smoothed\AHU_2_9_Blower_NDE_H_smoothed_roll200.csv
Smoothed plot: AHU_2_9_Blower_NDE_V.csv
  Saved: C:\Users\NGYX\Desktop\Murata_NGYX\data\smoothed\AHU_2_9_Blower_NDE_V_smoothed_roll200.csv
Smoothed plot: AHU_2_9_motor_DE_H.csv
  Saved: C:\Users\NGYX\Desktop\Murata_NGYX\data\smoothed\AHU_2_9_motor_DE_H_smoothed_roll200.csv
Smoothed plot: AH

In [13]:
import pandas as pd

# ============================================
# Load CSV
# ============================================
file_path = r"data/smoothed/AHU_2_9_motor_NDE_H_smoothed_roll200.csv"

df = pd.read_csv(file_path)

# ============================================
# Convert timestamp column
# ============================================
df["TIMESTAMP"] = pd.to_datetime(
    df["TIMESTAMP"],
    dayfirst=True,
    format="mixed"
)

# ============================================
# Sort chronologically
# ============================================
df = df.sort_values("TIMESTAMP")

# ============================================
# Train / test split (pick one)
# ============================================
# 1) Middle test window: set both TEST_START and TEST_END.
#    Test = rows with TEST_START <= TIMESTAMP < TEST_END.
#    Train = all other rows (before the window and after the window, one table each,
#    combined and sorted by time).
# 2) Tail test: set TEST_START = TEST_END = None and set SPLIT_AT.
#    Test = TIMESTAMP >= SPLIT_AT, train = strictly before.
# 3) Row ratio: set TEST_START, TEST_END, and SPLIT_AT all to None → 80% / 20% by row count.
TEST_START = None  # e.g. pd.Timestamp("2025-06-01 00:00:00")
TEST_END = None  # e.g. pd.Timestamp("2025-07-01 00:00:00")
SPLIT_AT = None

if (TEST_START is None) != (TEST_END is None):
    raise ValueError("Set both TEST_START and TEST_END for a middle window, or set both to None.")

if TEST_START is not None and TEST_END is not None:
    if TEST_START >= TEST_END:
        raise ValueError("TEST_START must be before TEST_END")
    in_window = (df["TIMESTAMP"] >= TEST_START) & (df["TIMESTAMP"] < TEST_END)
    test_df = df[in_window]
    train_df = pd.concat(
        [df[df["TIMESTAMP"] < TEST_START], df[df["TIMESTAMP"] >= TEST_END]],
        ignore_index=True,
    ).sort_values("TIMESTAMP")
    print("Split mode: middle test window", TEST_START, "->", TEST_END, "(end exclusive)")
elif SPLIT_AT is not None:
    train_df = df[df["TIMESTAMP"] < SPLIT_AT]
    test_df = df[df["TIMESTAMP"] >= SPLIT_AT]
    print("Split mode: tail test, TIMESTAMP >=", SPLIT_AT)
else:
    split_index = int(len(df) * 0.9)
    train_df = df.iloc[:split_index]
    test_df = df.iloc[split_index:]
    print("Split mode: 90% / 10% rows")

# ============================================
# Save split CSVs
# ============================================
train_df.to_csv("data_train_val.csv", index=False)
test_df.to_csv("data_test_anomalous.csv", index=False)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain period:")
print(train_df["TIMESTAMP"].min(), "->", train_df["TIMESTAMP"].max())

print("\nTest period:")
print(test_df["TIMESTAMP"].min(), "->", test_df["TIMESTAMP"].max())

Split mode: 90% / 10% rows
Train shape: (66061, 18)
Test shape: (7341, 18)

Train period:
2023-01-09 00:04:00 -> 2025-12-25 01:59:00

Test period:
2025-12-25 02:04:00 -> 2026-12-01 23:56:00
